# BigQuant single-factor submission

This notebook is generated for exactly one factor submission. Upload this `.ipynb` file alone.


In [ ]:
def main(datasources, start_date, end_date):
    """Build a persistent, near-touch weighted order-book imbalance factor."""

    import numpy as np
    import pandas as pd
    import dai

    if not isinstance(datasources, dict) or "bar1m" not in datasources:
        raise ValueError("datasources must contain the logical source 'bar1m'")

    bar1m_table = datasources["bar1m"]
    if not isinstance(bar1m_table, str) or not bar1m_table:
        raise ValueError("datasources['bar1m'] must be a non-empty table name")

    # Near-touch queues receive larger weights than distant queues.
    sql = f"""
    WITH minute_book AS (
        SELECT
            date,
            instrument,
            bid_volume1::DOUBLE
                + 0.5000 * bid_volume2::DOUBLE
                + 0.2500 * bid_volume3::DOUBLE
                + 0.1250 * bid_volume4::DOUBLE
                + 0.0625 * bid_volume5::DOUBLE AS weighted_bid,
            ask_volume1::DOUBLE
                + 0.5000 * ask_volume2::DOUBLE
                + 0.2500 * ask_volume3::DOUBLE
                + 0.1250 * ask_volume4::DOUBLE
                + 0.0625 * ask_volume5::DOUBLE AS weighted_ask
        FROM {bar1m_table}
        WHERE instrument IS NOT NULL
        AND bid_volume1 >= 0 AND bid_volume2 >= 0 AND bid_volume3 >= 0
        AND bid_volume4 >= 0 AND bid_volume5 >= 0
        AND ask_volume1 >= 0 AND ask_volume2 >= 0 AND ask_volume3 >= 0
        AND ask_volume4 >= 0 AND ask_volume5 >= 0
    ),
    minute_signal AS (
        SELECT
            date,
            instrument,
            weighted_bid,
            weighted_ask,
            CASE
                WHEN weighted_bid > weighted_ask THEN 1.0
                WHEN weighted_bid < weighted_ask THEN -1.0
                ELSE 0.0
            END AS direction
        FROM minute_book
        WHERE weighted_bid + weighted_ask > 0
    ),
    daily_signal AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            SUM(weighted_bid - weighted_ask)
                / (SUM(weighted_bid + weighted_ask) + 1.0) AS pressure,
            ABS(AVG(direction)) AS persistence
        FROM minute_signal
        GROUP BY date::DATE, instrument
    )
    SELECT
        date,
        instrument,
        pressure * persistence AS factor
    FROM daily_signal
    """
    factor_df = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    pool_df = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    required_factor_columns = {"date", "instrument", "factor"}
    required_pool_columns = {"date", "instrument"}
    if not required_factor_columns.issubset(factor_df.columns):
        missing = sorted(required_factor_columns - set(factor_df.columns))
        raise ValueError(f"factor query is missing columns: {missing}")
    if not required_pool_columns.issubset(pool_df.columns):
        missing = sorted(required_pool_columns - set(pool_df.columns))
        raise ValueError(f"instrument query is missing columns: {missing}")

    factor_df = factor_df[["date", "instrument", "factor"]].copy()
    pool_df = pool_df[["date", "instrument"]].copy()

    for frame in (factor_df, pool_df):
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)

    factor_df["factor"] = pd.to_numeric(factor_df["factor"], errors="coerce")
    factor_df["factor"] = factor_df["factor"].replace([np.inf, -np.inf], np.nan)
    factor_df = factor_df.dropna(subset=["date", "instrument"])
    factor_df = factor_df.drop_duplicates(["date", "instrument"], keep="last")

    pool_df = pool_df.dropna(subset=["date", "instrument"])
    pool_df = pool_df.drop_duplicates(["date", "instrument"], keep="last")

    out = pool_df.merge(factor_df, on=["date", "instrument"], how="left")
    out = out[["date", "instrument", "factor"]]
    return out.sort_values(["date", "instrument"]).reset_index(drop=True)
